In [4]:
# 1. INSTALLATION ET IMPORTS
!pip install xgboost mlflow

import pandas as pd
import numpy as np
import xgboost as xgb
import mlflow
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# 2. CHARGEMENT ET PREPROCESSING LÉGER (CORRIGÉ)
df = pd.read_csv('LengthOfStay.csv')

# Transformation des colonnes connues
df['gender'] = df['gender'].map({'F': 0, 'M': 1})
df['rcount'] = df['rcount'].replace('5+', '5').astype(int)

# --- CORRECTION ICI ---
# On retire 'eid' ET les colonnes de dates 'vdate', 'discharged'
# car elles ne sont pas numériques et causent l'erreur XGBoost.
cols_to_drop = ['eid', 'vdate', 'discharged', 'lengthofstay']

# On prépare les dummies pour 'facid'
df_processed = pd.get_dummies(df, columns=['facid'], drop_first=True)

# Définition de X et y
X = df_processed.drop(columns=cols_to_drop)
y = df_processed['lengthofstay']
# ----------------------

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vérification rapide des types (Optionnel mais rassurant)
print(X.dtypes.unique()) # Doit afficher uniquement des types numériques

# 3. CONFIGURATION MLFLOW
# En local, cela créera un dossier ./mlruns
mlflow.set_experiment("Hospital_LOS_Prediction")

# Active l'enregistrement automatique (métriques, paramètres, modèle)
mlflow.xgboost.autolog()

with mlflow.start_run(run_name="XGBoost_Final_Model"):
    
    # 4. ENTRAÎNEMENT
    # Note : XGBoost gère nativement les NaN si tu en as dans ton dataset
    model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    # 5. ÉVALUATION
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    print(f"✅ Modèle entraîné. MAE: {mae:.2f}, R2: {r2:.2f}")
    
    # On peut ajouter des tags pour mieux s'y retrouver dans l'UI MLflow
    mlflow.set_tag("release.version", "1.0.0")
    mlflow.set_tag("owner", "toto")

# 6. POUR LANCER L'INTERFACE MLFLOW (à taper dans ton terminal local)
# !mlflow ui


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
[dtype('int64') dtype('float64') dtype('bool')]


2026/03/16 16:26:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/cyril/projets/cmv/mlflow/venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/03/16 16:26:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/16 16:26:13 WARNING 

✅ Modèle entraîné. MAE: 0.36, R2: 0.96
